# Data Cleaning

In [1]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt
BASE = Path.cwd()
if not (BASE / "evidence").exists():
    BASE = BASE / "notebooks" 
assert (BASE / "evidence").is_dir(), "Open from the project root or notebooks."
EVIDENCE = BASE / "evidence"
def read_json(name):
    return json.loads((EVIDENCE / name).read_text(encoding="utf-8-sig"))
pd.set_option("display.max_colwidth", 90)
plt.rcParams.update({"figure.figsize": (10, 4.5), "axes.spines.top": False,
                     "axes.spines.right": False, "font.size": 11})


## Transformation sequence
Source snapshots → parse values and dates → classify Calendar events → match services to sales/programmes → attach location and weather → derive outcome windows and predictors → validate and publish.

The production transformations are SQL/dbt and Python scripts. This chapter contains their exact SQL and archived model-input checks, rather than a second implementation with different rules.

In [2]:
cleaning = pd.DataFrame([
    ["Money and dates", "Keep original values; parse analytical types; preserve invalid/missing values", "dbt/models/sales.sql; payments.sql; recurring_payments.sql"],
    ["Calendar classification", "Separate cancelled, administrative, parked, scheduled and corrective-return events", "dbt/models/calendar_events.sql"],
    ["Temporary placeholders", "Use exact owner-confirmed event and time; never classify every Sunday as parked", "scripts/calendar_appointment_policy.py"],
    ["Programme linkage", "Link confirmed 1x + Upsell 2x or combined programmes; preserve separate sale rows", "scripts/service_programme_policy.py; dbt/models/calendar_event_matches.sql"],
    ["Payment reminders", "Dates are reminders; green only means the date passed; paid totals come from recorded amounts", "scripts/sales_payment_rules.py"],
    ["Location matching", "Queue ambiguous addresses; do not replace missing precise locations with a false exact point", "scripts/cp2_geocoding.py"],
    ["Time availability", "Only attach information available by prediction time", "scripts/callback_live_features.py; callback_live_weather.py"]
], columns=["Step", "Rule", "Implementation"])
display(cleaning)

Step,Rule,Implementation
Money and dates,Keep original values; parse analytical types; preserve invalid/missing values,dbt/models/sales.sql; payments.sql; recurring_payments.sql
Calendar classification,"Separate cancelled, administrative, parked, scheduled and corrective-return events",dbt/models/calendar_events.sql
Temporary placeholders,Use exact owner-confirmed event and time; never classify every Sunday as parked,scripts/calendar_appointment_policy.py
Programme linkage,Link confirmed 1x + Upsell 2x or combined programmes; preserve separate sale rows,scripts/service_programme_policy.py; dbt/models/calendar_event_matches.sql
Payment reminders,Dates are reminders; green only means the date passed; paid totals come from recorded amounts,scripts/sales_payment_rules.py
Location matching,Queue ambiguous addresses; do not replace missing precise locations with a false exact point,scripts/cp2_geocoding.py
Time availability,Only attach information available by prediction time,scripts/callback_live_features.py; callback_live_weather.py


## Sales transformation
The following is copied from the actual dbt model. The full file is retained in `evidence/sql/sales.sql`; source hashes are listed in `source_manifest.json`.

In [3]:
sales_sql = (EVIDENCE / "sql/sales.sql").read_text(encoding="utf-8-sig")
print("\n".join(sales_sql.splitlines()[:65]))

{{ config(tags=['operational']) }}
with prepared as (
    select *,
        {{ sales_reference_array('source_column_007') }} as parsed_sales_ids,
        {{ rm_decimal('source_column_028') }} as sale_total_rm
    from {{ source('operational_bronze', 'sales') }}
)
select *,
    {{ sales_remaining_raw_aliases() }}
    '{{ source("operational_bronze", "sales").identifier }}' as snapshot_table,
    concat('{{ source("operational_bronze", "sales").identifier }}', ':', cast(source_sheet_row as string)) as sales_row_id,
    source_column_007 as customer_id_raw,
    case when array_length(parsed_sales_ids) = 1 then parsed_sales_ids[offset(0)] end as sales_record_id,
    trim(source_column_007) != '' as id_present_candidate,
    trim(source_column_007) != '' and array_length(parsed_sales_ids) != 1 as id_needs_review,
    trim(source_column_028) != '' and sale_total_rm is null as total_needs_review,
    regexp_contains(upper(source_column_008), r'\b(?:OLD|RETURNING)\b') as recorded_returning_cli

## Calendar transformation
Cancellation and administrative rules take priority. A moved temporary placeholder expires its exact-slot override. A rescheduled event is not an additional completed paid visit. Event identity is retained independently of changing source row numbers.

In [4]:
calendar_sql = (EVIDENCE / "sql/calendar_events.sql").read_text(encoding="utf-8-sig")
print("\n".join(calendar_sql.splitlines()[:48]))

{{ config(tags=['calendar']) }}
with raw_events as (
    select *,
        upper(coalesce(summary, '')) as summary_upper,
        safe_cast(regexp_extract(summary, r'\b(\d{1,2})\s*/\s*\d{1,2}\b') as int64) as session_current,
        safe_cast(regexp_extract(summary, r'\b\d{1,2}\s*/\s*(\d{1,2})\b') as int64) as session_total
    from {{ source('calendar_bronze', 'events') }}
), classified as (
    select *,
        {{ calendar_event_category('summary_upper', 'status', 'session_current', 'session_total') }}
            as rule_event_category,
        {{ calendar_event_category('summary_upper', 'status', 'session_current', 'session_total', false) }}
            as non_claim_event_category
    from raw_events
), reviewed as (
    select c.*,
        r.review_id as warranty_label_review_id,
        r.review_label as warranty_label_review_status,
        r.reviewed_at as warranty_label_reviewed_at,
        case
            -- Cancellation/admin rules still control operational eligibility.
 

## Archived model-input checks
These are the recorded checks on the 5,695-row frozen-refit input, not a fresh audit of today's Google records. Two weather-date values were missing; missing coverage is retained as a limitation.

In [5]:
checks = read_json("source_checks.json")
display(pd.DataFrame([{"Check":k.replace("_"," "), "Recorded count":v} for k,v in checks.items()]))
assert checks["duplicate_service_keys"] == 0
assert checks["unresolved_labels"] == 0
assert checks["non_30_day_windows"] == 0
assert checks["outcomes_after_observation"] == 0
assert checks["weather_on_or_after_service"] == 0

Check,Recorded count
unresolved labels,0
duplicate service keys,0
non 30 day windows,0
outcomes after observation,0
weather on or after service,0
weather latest date missing,2


## Reproduce from the private input
From the project root, `python -m scripts.build_supervisor_model_review` reloads the hash-pinned input, calls `benchmark_warranty_models.load_dataset_input` and `compare_callback_candidates_v2.prepare_input`, and computes these aggregate checks. It writes review artifacts only; it does not publish or retrain.

Production SQL checks run through `scripts/cp2_reporting_release.py`; run its help before using release options. Do not run every script in the folder: some publish, renew credentials or evaluate an experiment once.